# Raw 20 ms SBP Factor Analysis: T12 chronological session holdout

This notebook asks whether the **shared covariance** of instantaneous 20 ms × 128-channel SBP activity is lower-dimensional than raw PCA suggested. Factor Analysis (FA) separates a low-rank shared covariance from diagonal channel-private variance.

It reuses the exact per-session sufficient statistics produced by `raw_20ms_sbp_pca.ipynb`; it does not reread the neural cache, use phoneme labels, infer timings, or use decoder representations.

## Evaluation contract

- First 16 chronological sessions: candidate-dimension fitting.
- Next 4 earlier sessions: dimension selection by pooled average Gaussian log likelihood.
- Final 4 future sessions: untouched final evaluation.
- Primary normalization: session-wise z-scoring (transductive nuisance removal).
- Control: training-global z-scoring (inductive, preserving future-day drift).

The maximum-validation-likelihood factor count is the primary selection. The notebook also reports the smallest candidate attaining 95% of the validation likelihood gain over a diagonal Gaussian, because millions of correlated bins can make tiny improvements favor large models.

Run smoke mode first. Then set `SMOKE_MODE = False` to consume the completed full PCA statistics.


In [ ]:
# Mount Drive, clone/update the repository, and install only missing packages.

from pathlib import Path
import importlib.util
import os
import subprocess
import sys

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/ethan-read/utah-ssl.git'
REPO_DIR = Path('/content/utah-ssl') if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        status = subprocess.run(
            ['git', '-C', str(REPO_DIR), 'status', '--porcelain'],
            check=True, capture_output=True, text=True,
        )
        if status.stdout.strip():
            print('Using existing checkout with local changes; automatic pull skipped.')
        else:
            subprocess.run(
                ['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'],
                check=True,
            )

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

required_packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'sklearn': 'scikit-learn',
}
missing = [pip_name for module, pip_name in required_packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

print({'in_colab': IN_COLAB, 'repo_dir': str(REPO_DIR), 'installed': missing})


In [ ]:
# Experiment configuration.

import json
import tempfile
import warnings
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.decomposition import FactorAnalysis
from sklearn.exceptions import ConvergenceWarning

SMOKE_MODE = True
OVERWRITE_OUTPUT = False

N_CHANNELS = 128
EPSILON = 1e-8
FA_INITIAL_PRIVATE_VARIANCE_FLOOR = 1e-12
FA_CANDIDATE_DIMS = (0, 1, 2, 4, 6, 8, 12, 16, 24, 32, 48, 64, 80, 96, 112)
FA_MAX_ITER = 2000
FA_TOL = 1e-5
PARSIMONY_GAIN_FRACTION = 0.95
N_CV_SESSIONS = 4

EXPECTED_HOLDOUT_SESSION_IDS = (
    't12.2022.08.13',
    't12.2022.08.18',
    't12.2022.08.23',
    't12.2022.08.25',
)

DRIVE_ROOT = Path('/content/drive/MyDrive') if IN_COLAB else Path('/Users/home/My Drive')
UTAH_SSL_ROOT = DRIVE_ROOT / 'utah_ssl'
OUTPUT_ROOT = UTAH_SSL_ROOT / 'outputs' / 'neural_trajectories'

PCA_BASE_RUN_NAME = 'raw_20ms_sbp_pca_t12_chronological_v1'
PCA_RUN_NAME = PCA_BASE_RUN_NAME + ('_smoke' if SMOKE_MODE else '')
PCA_ARTIFACT_DIR = OUTPUT_ROOT / PCA_RUN_NAME

BASE_RUN_NAME = 'raw_20ms_sbp_factor_analysis_t12_chronological_v1'
RUN_NAME = BASE_RUN_NAME + ('_smoke' if SMOKE_MODE else '')
OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
if OUTPUT_DIR.exists() and not OVERWRITE_OUTPUT:
    raise FileExistsError(
        f'Refusing to start because output already exists: {OUTPUT_DIR}. '
        'Set OVERWRITE_OUTPUT=True only after reviewing it.'
    )
if OUTPUT_DIR.exists():
    print('Existing output will be moved to a timestamped backup after analysis:', OUTPUT_DIR)
WORK_DIR = Path(tempfile.mkdtemp(prefix=f'{RUN_NAME}_'))

if any(q < 0 or q >= N_CHANNELS for q in FA_CANDIDATE_DIMS):
    raise ValueError('FA candidate dimensions must be in [0, N_CHANNELS)')
if tuple(sorted(set(FA_CANDIDATE_DIMS))) != FA_CANDIDATE_DIMS:
    raise ValueError('FA_CANDIDATE_DIMS must be strictly increasing and unique')
if FA_CANDIDATE_DIMS[0] != 0:
    raise ValueError('Candidate grid must include q=0 as the diagonal baseline')

print({
    'smoke_mode': SMOKE_MODE,
    'pca_artifact_dir': str(PCA_ARTIFACT_DIR),
    'output_dir': str(OUTPUT_DIR),
    'candidate_dims': FA_CANDIDATE_DIMS,
})


## Reopen and audit the PCA sufficient statistics

The source artifact is the permanent data record for this analysis. The checks below enforce the same native 20 ms, first-128-channel clipped-FP16 SBP contract and the same future-session holdout. Smoke and full artifacts cannot be mixed.


In [ ]:
# Load the exact per-session sufficient statistics and validate provenance.

import hashlib

required_source_artifacts = (
    'config.json',
    'provenance.json',
    'cache_audit.json',
    'sufficient_statistics.npz',
    'session_inventory.csv',
    'summary.json',
)
missing_source = [name for name in required_source_artifacts
                  if not (PCA_ARTIFACT_DIR / name).is_file()]
if missing_source:
    raise FileNotFoundError(
        f'PCA source artifact is incomplete at {PCA_ARTIFACT_DIR}: {missing_source}'
    )

source_config = json.loads((PCA_ARTIFACT_DIR / 'config.json').read_text())
source_provenance = json.loads((PCA_ARTIFACT_DIR / 'provenance.json').read_text())
source_summary = json.loads((PCA_ARTIFACT_DIR / 'summary.json').read_text())
source_inventory = pd.read_csv(PCA_ARTIFACT_DIR / 'session_inventory.csv')
if source_summary.get('run_name') != PCA_RUN_NAME:
    raise ValueError(f'PCA source summary has the wrong run name: {source_summary.get("run_name")!r}')

if bool(source_config.get('smoke_mode')) != SMOKE_MODE:
    raise ValueError('PCA artifact smoke/full mode does not match this run')
expected_contract = {
    'dataset': 'brain2text24',
    'source_split': 'competition_train',
    'subject_id': 't12',
    'bin_size_ms': 20,
    'cache_variant': 'cache_v1_sbpclip12500_fp16_raw',
    'smoothing': 'none',
}
for key, expected in expected_contract.items():
    if source_config.get(key) != expected:
        raise ValueError(f'PCA source contract mismatch for {key}: {source_config.get(key)!r}')

signal_spec = source_config.get('signal_spec') or {}
if signal_spec.get('mode') != 'sbp_only':
    raise ValueError(f'Expected SBP-only source SignalSpec: {signal_spec}')
if int(signal_spec.get('sbp_dim', -1)) != N_CHANNELS:
    raise ValueError(f'Expected {N_CHANNELS} SBP channels: {signal_spec}')
if int(signal_spec.get('column_start', -1)) != 0:
    raise ValueError(f'Expected first area-6v channel at column 0: {signal_spec}')
if signal_spec.get('missing_channel_policy') != 'error':
    raise ValueError(f'Expected missing-channel policy error: {signal_spec}')

fit_session_ids = tuple(source_config['fit_session_ids'])
heldout_session_ids = tuple(source_config['heldout_session_ids'])
if len(fit_session_ids) != 20 or len(heldout_session_ids) != 4:
    raise ValueError('Expected the PCA 20-session fit / 4-session holdout split')
if heldout_session_ids != EXPECTED_HOLDOUT_SESSION_IDS:
    raise ValueError(f'Unexpected future sessions: {heldout_session_ids}')
if fit_session_ids[-1] != 't12.2022.08.11':
    raise ValueError(f'Unexpected final pre-holdout session: {fit_session_ids[-1]}')
if set(fit_session_ids) & set(heldout_session_ids):
    raise AssertionError('Fit and future-session partitions overlap')

cv_train_session_ids = fit_session_ids[:-N_CV_SESSIONS]
cv_validation_session_ids = fit_session_ids[-N_CV_SESSIONS:]
if len(cv_train_session_ids) != 16 or len(cv_validation_session_ids) != 4:
    raise AssertionError('Expected a 16-session CV fit / 4-session validation split')

with np.load(PCA_ARTIFACT_DIR / 'sufficient_statistics.npz') as payload:
    stats_session_ids = tuple(str(value) for value in payload['session_ids'].tolist())
    stats_n = np.asarray(payload['n'], dtype=np.int64)
    stats_sum = np.asarray(payload['sum'], dtype=np.float64)
    stats_sum_sq = np.asarray(payload['sum_sq'], dtype=np.float64)
    stats_cross = np.asarray(payload['cross'], dtype=np.float64)

all_session_ids = fit_session_ids + heldout_session_ids
if stats_session_ids != all_session_ids:
    raise AssertionError('Sufficient-statistics sessions do not match the serialized split')
if stats_n.shape != (24,) or stats_sum.shape != (24, N_CHANNELS):
    raise ValueError('Unexpected sufficient-statistics vector shapes')
if stats_cross.shape != (24, N_CHANNELS, N_CHANNELS):
    raise ValueError('Unexpected sufficient-statistics cross-product shape')
if np.any(stats_n <= 1):
    raise ValueError('Every session needs at least two bins')
if not all(np.isfinite(x).all() for x in (stats_sum, stats_sum_sq, stats_cross)):
    raise ValueError('Source sufficient statistics contain nonfinite values')
required_inventory_columns = {'session_id', 'partition', 'trials', 'bins'}
if not required_inventory_columns <= set(source_inventory.columns):
    raise ValueError('Source session inventory is missing required columns')
inventory_by_session = source_inventory.set_index('session_id')
if tuple(source_inventory['session_id']) != stats_session_ids:
    raise AssertionError('Source inventory session order does not match sufficient statistics')
if not np.array_equal(inventory_by_session.loc[list(stats_session_ids), 'bins'].to_numpy(), stats_n):
    raise AssertionError('Source inventory bin counts do not match sufficient statistics')

session_stats = {
    session_id: {
        'n': int(stats_n[index]),
        'sum': stats_sum[index].copy(),
        'sum_sq': stats_sum_sq[index].copy(),
        'cross': stats_cross[index].copy(),
    }
    for index, session_id in enumerate(stats_session_ids)
}

source_hashes = {
    name: hashlib.sha256((PCA_ARTIFACT_DIR / name).read_bytes()).hexdigest()
    for name in required_source_artifacts
}
git_commit = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=str(REPO_DIR), check=True,
    capture_output=True, text=True,
).stdout.strip()
git_status = subprocess.run(
    ['git', 'status', '--porcelain'], cwd=str(REPO_DIR), check=True,
    capture_output=True, text=True,
).stdout.splitlines()

config = {
    'run_name': RUN_NAME,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'smoke_mode': SMOKE_MODE,
    'source_pca_run_name': PCA_RUN_NAME,
    'source_pca_artifact_dir': str(PCA_ARTIFACT_DIR),
    'source_artifact_sha256': source_hashes,
    'dataset': source_config['dataset'],
    'source_split': source_config['source_split'],
    'subject_id': source_config['subject_id'],
    'bin_size_ms': source_config['bin_size_ms'],
    'cache_variant': source_config['cache_variant'],
    'signal_spec': signal_spec,
    'cv_train_session_ids': list(cv_train_session_ids),
    'cv_validation_session_ids': list(cv_validation_session_ids),
    'final_fit_session_ids': list(fit_session_ids),
    'future_heldout_session_ids': list(heldout_session_ids),
    'normalization_conditions': source_config['normalization_conditions'],
    'fa_candidate_dims': list(FA_CANDIDATE_DIMS),
    'fa_max_iter': FA_MAX_ITER,
    'fa_tolerance': FA_TOL,
    'fa_initial_private_variance_floor': FA_INITIAL_PRIVATE_VARIANCE_FLOOR,
    'selection_metric': 'pooled validation average Gaussian log likelihood per bin',
    'parsimony_gain_fraction': PARSIMONY_GAIN_FRACTION,
    'bin_weighting': 'equal weight per native 20 ms bin',
    'implementation': 'sklearn.decomposition.FactorAnalysis fit exactly from covariance pseudo-observations',
    'sklearn_version': sklearn.__version__,
    'git_commit': git_commit,
    'git_worktree_dirty': bool(git_status),
}
(WORK_DIR / 'config.json').write_text(json.dumps(config, indent=2) + '\n')

provenance = {
    'classification': 'repository-native follow-up analysis',
    'input_artifact': str(PCA_ARTIFACT_DIR),
    'input_artifact_sha256': source_hashes,
    'data_reuse': (
        'Exact per-session sufficient statistics from the validated raw 20 ms SBP PCA run; '
        'no neural arrays were reread and no labels or decoder outputs were used.'
    ),
    'method': (
        'Gaussian Factor Analysis with diagonal private noise, using scikit-learn. '
        'Covariance pseudo-observations preserve the empirical second moment exactly.'
    ),
    'ai_assistance': (
        'Codex designed and generated this notebook; human review and empirical '
        'interpretation remain required.'
    ),
}
(WORK_DIR / 'provenance.json').write_text(json.dumps(provenance, indent=2) + '\n')
source_inventory.to_csv(WORK_DIR / 'source_session_inventory.csv', index=False)

print({
    'cv_train_sessions': cv_train_session_ids,
    'cv_validation_sessions': cv_validation_session_ids,
    'future_sessions': heldout_session_ids,
    'total_bins': int(stats_n.sum()),
})


## Covariance and Factor Analysis helpers

FA is a covariance model: \(C = W W^T + \Psi\), where \(W W^T\) is shared covariance and \(\Psi\) is diagonal private variance. The covariance pseudo-observations below preserve the empirical second moment exactly, so fitting them gives the same Gaussian FA objective as fitting every original bin, without materializing those bins.


In [ ]:
# Sufficient-statistics and normalization helpers.

def empty_sufficient_stats(dim):
    return {
        'n': 0,
        'sum': np.zeros(dim, dtype=np.float64),
        'sum_sq': np.zeros(dim, dtype=np.float64),
        'cross': np.zeros((dim, dim), dtype=np.float64),
    }


def combine_sufficient_stats(items):
    items = list(items)
    if not items:
        raise ValueError('Cannot combine an empty statistics collection')
    result = empty_sufficient_stats(items[0]['sum'].shape[0])
    for item in items:
        result['n'] += int(item['n'])
        result['sum'] += item['sum']
        result['sum_sq'] += item['sum_sq']
        result['cross'] += item['cross']
    return result


def mean_and_std(stats, epsilon=EPSILON):
    mean = stats['sum'] / stats['n']
    variance = stats['sum_sq'] / stats['n'] - np.square(mean)
    std = np.sqrt(np.maximum(variance, 0.0))
    return mean, np.maximum(std, epsilon)


def scatter_about_center(stats, center):
    center = np.asarray(center, dtype=np.float64)
    return (
        stats['cross']
        - np.outer(center, stats['sum'])
        - np.outer(stats['sum'], center)
        + stats['n'] * np.outer(center, center)
    )


def scale_scatter(scatter, std):
    return np.asarray(scatter, dtype=np.float64) / np.outer(std, std)


def session_zscore_scatter(stats):
    mean, std = mean_and_std(stats)
    return scale_scatter(scatter_about_center(stats, mean), std)


def session_normalized_moment(session_ids):
    scatter = sum(
        (session_zscore_scatter(session_stats[s]) for s in session_ids),
        start=np.zeros((N_CHANNELS, N_CHANNELS), dtype=np.float64),
    )
    n = sum(session_stats[s]['n'] for s in session_ids)
    return 0.5 * (scatter + scatter.T) / n, n


def global_normalized_moment(session_ids, normalization_session_ids):
    normalization_stats = combine_sufficient_stats(
        session_stats[s] for s in normalization_session_ids
    )
    center, std = mean_and_std(normalization_stats)
    evaluation_stats = combine_sufficient_stats(session_stats[s] for s in session_ids)
    scatter = scale_scatter(scatter_about_center(evaluation_stats, center), std)
    scatter = 0.5 * (scatter + scatter.T)
    return scatter / evaluation_stats['n'], evaluation_stats['n']


def validate_moment(moment, label):
    moment = 0.5 * (np.asarray(moment) + np.asarray(moment).T)
    if moment.shape != (N_CHANNELS, N_CHANNELS) or not np.isfinite(moment).all():
        raise ValueError(f'{label}: invalid second moment')
    tolerance = max(1.0, float(np.max(np.abs(moment)))) * 1e-9
    if np.linalg.eigvalsh(moment).min() < -tolerance:
        raise ValueError(f'{label}: second moment is not positive semidefinite')
    return moment


In [ ]:
# Exact covariance-based FA fitting and evaluation helpers.

LOG_2PI = float(np.log(2.0 * np.pi))


def covariance_pseudo_observations(moment):
    moment = 0.5 * (np.asarray(moment, dtype=np.float64) + np.asarray(moment).T)
    eigenvalues, eigenvectors = np.linalg.eigh(moment)
    tolerance = max(1.0, float(np.max(np.abs(eigenvalues)))) * 1e-10
    if eigenvalues.min() < -tolerance:
        raise ValueError(f'Moment has negative eigenvalue {eigenvalues.min()}')
    eigenvalues = np.maximum(eigenvalues, 0.0)
    root_rows = np.sqrt(eigenvalues)[:, None] * eigenvectors.T
    pseudo = np.vstack((root_rows, -root_rows)) * np.sqrt(moment.shape[0])
    np.testing.assert_allclose(
        pseudo.T @ pseudo / len(pseudo), moment, rtol=2e-10, atol=2e-10,
    )
    return pseudo


def fit_factor_model(moment, q):
    moment = validate_moment(moment, f'FA q={q}')
    if q == 0:
        noise = np.maximum(np.diag(moment), FA_INITIAL_PRIVATE_VARIANCE_FLOOR)
        return {
            'q': 0,
            'loadings': np.zeros((N_CHANNELS, 0), dtype=np.float64),
            'noise_variance': noise,
            'n_iter': 0,
            'converged': True,
        }
    estimator = FactorAnalysis(
        n_components=q,
        tol=FA_TOL,
        max_iter=FA_MAX_ITER,
        svd_method='lapack',
        noise_variance_init=np.maximum(
            np.diag(moment), FA_INITIAL_PRIVATE_VARIANCE_FLOOR
        ),
    )
    with warnings.catch_warnings(record=True) as caught_warnings:
        warnings.simplefilter('always', ConvergenceWarning)
        estimator.fit(covariance_pseudo_observations(moment))
    convergence_warnings = [
        warning for warning in caught_warnings
        if issubclass(warning.category, ConvergenceWarning)
    ]
    if convergence_warnings:
        raise RuntimeError(
            f'FA did not converge for q={q} after {estimator.n_iter_} iterations'
        )
    loadings = np.asarray(estimator.components_.T, dtype=np.float64)
    noise = np.asarray(estimator.noise_variance_, dtype=np.float64)
    if loadings.shape != (N_CHANNELS, q) or not np.isfinite(loadings).all():
        raise RuntimeError(f'FA returned invalid loadings for q={q}')
    if noise.shape != (N_CHANNELS,) or not np.isfinite(noise).all() or np.any(noise <= 0):
        raise RuntimeError(f'FA returned invalid private variances for q={q}')
    return {
        'q': int(q),
        'loadings': loadings,
        'noise_variance': noise,
        'n_iter': int(estimator.n_iter_),
        'converged': True,
    }


def model_covariance(model):
    loadings = model['loadings']
    covariance = loadings @ loadings.T + np.diag(model['noise_variance'])
    return 0.5 * (covariance + covariance.T)


def average_log_likelihood(moment, model):
    covariance = model_covariance(model)
    sign, logdet = np.linalg.slogdet(covariance)
    if sign <= 0 or not np.isfinite(logdet):
        raise ValueError('FA covariance is not positive definite')
    trace_term = float(np.trace(np.linalg.solve(covariance, moment)))
    return float(-0.5 * (N_CHANNELS * LOG_2PI + logdet + trace_term))


def shared_variance_fraction(model):
    shared = model['loadings'] @ model['loadings'].T
    return float(np.trace(shared) / np.trace(model_covariance(model)))


def posterior_reconstruction_mse(moment, model):
    covariance = model_covariance(model)
    shared = model['loadings'] @ model['loadings'].T
    reconstruction = np.linalg.solve(covariance, shared).T
    residual = np.eye(N_CHANNELS) - reconstruction
    mse = np.trace(residual @ moment @ residual.T) / N_CHANNELS
    return float(max(mse, 0.0))


def ordered_loadings(model):
    loadings = model['loadings'].copy()
    if loadings.shape[1] == 0:
        return loadings
    order = np.argsort(np.sum(np.square(loadings), axis=0))[::-1]
    loadings = loadings[:, order]
    for index in range(loadings.shape[1]):
        pivot = int(np.argmax(np.abs(loadings[:, index])))
        if loadings[pivot, index] < 0:
            loadings[:, index] *= -1
    return loadings


In [ ]:
# Synthetic checks: covariance compression and likelihood scoring must match direct samples.

rng = np.random.default_rng(11)
synthetic_latent = rng.normal(size=(700, 3))
synthetic_mixing = rng.normal(size=(3, 12))
synthetic = synthetic_latent @ synthetic_mixing + rng.normal(
    scale=np.linspace(0.25, 0.8, 12), size=(700, 12),
)
synthetic -= synthetic.mean(axis=0)
synthetic_moment = synthetic.T @ synthetic / len(synthetic)

pseudo = covariance_pseudo_observations(synthetic_moment)
np.testing.assert_allclose(
    pseudo.T @ pseudo / len(pseudo), synthetic_moment, rtol=2e-10, atol=2e-10,
)

# Use the generic scorer at 12 dimensions for this independent numerical check.
old_n_channels = N_CHANNELS
N_CHANNELS = synthetic.shape[1]
synthetic_model = fit_factor_model(synthetic_moment, 3)
covariance = model_covariance(synthetic_model)
covariance_inverse = np.linalg.inv(covariance)
_, logdet = np.linalg.slogdet(covariance)
direct_scores = -0.5 * (
    N_CHANNELS * LOG_2PI + logdet
    + np.einsum('ni,ij,nj->n', synthetic, covariance_inverse, synthetic)
)
np.testing.assert_allclose(
    average_log_likelihood(synthetic_moment, synthetic_model),
    direct_scores.mean(), rtol=1e-10, atol=1e-10,
)
if average_log_likelihood(synthetic_moment, synthetic_model) <= average_log_likelihood(
    synthetic_moment, fit_factor_model(synthetic_moment, 0)
):
    raise AssertionError('Synthetic shared-factor model did not beat the diagonal baseline')
if not synthetic_model['converged']:
    raise AssertionError('Synthetic FA convergence check failed')
N_CHANNELS = old_n_channels

print('Synthetic covariance-FA and direct likelihood checks passed.')


## Build the nested chronological evaluation

Session-wise normalization uses each evaluation session's own unlabeled mean and standard deviation and is therefore transductive. Global normalization uses only the current fitting partition: sessions 1–16 during dimension selection and sessions 1–20 for final future-session evaluation.


In [ ]:
# Construct fit, validation, and future moments without leakage.

condition_moments = {}
for condition in ('session_zscore', 'train_global_zscore'):
    if condition == 'session_zscore':
        cv_train_moment, cv_train_n = session_normalized_moment(cv_train_session_ids)
        cv_validation_moments = {
            s: session_normalized_moment((s,)) for s in cv_validation_session_ids
        }
        cv_validation_pooled = session_normalized_moment(cv_validation_session_ids)
        final_train_moment, final_train_n = session_normalized_moment(fit_session_ids)
        future_moments = {s: session_normalized_moment((s,)) for s in heldout_session_ids}
        future_pooled = session_normalized_moment(heldout_session_ids)
    else:
        cv_train_moment, cv_train_n = global_normalized_moment(
            cv_train_session_ids, cv_train_session_ids,
        )
        cv_validation_moments = {
            s: global_normalized_moment((s,), cv_train_session_ids)
            for s in cv_validation_session_ids
        }
        cv_validation_pooled = global_normalized_moment(
            cv_validation_session_ids, cv_train_session_ids,
        )
        final_train_moment, final_train_n = global_normalized_moment(
            fit_session_ids, fit_session_ids,
        )
        future_moments = {
            s: global_normalized_moment((s,), fit_session_ids)
            for s in heldout_session_ids
        }
        future_pooled = global_normalized_moment(heldout_session_ids, fit_session_ids)

    condition_moments[condition] = {
        'cv_train': (validate_moment(cv_train_moment, f'{condition} cv train'), cv_train_n),
        'cv_validation': {
            s: (validate_moment(moment, f'{condition} cv {s}'), n)
            for s, (moment, n) in cv_validation_moments.items()
        },
        'cv_validation_pooled': (
            validate_moment(cv_validation_pooled[0], f'{condition} cv pooled'),
            cv_validation_pooled[1],
        ),
        'final_train': (
            validate_moment(final_train_moment, f'{condition} final train'), final_train_n,
        ),
        'future': {
            s: (validate_moment(moment, f'{condition} future {s}'), n)
            for s, (moment, n) in future_moments.items()
        },
        'future_pooled': (
            validate_moment(future_pooled[0], f'{condition} future pooled'),
            future_pooled[1],
        ),
    }

assert not (set(cv_train_session_ids) & set(cv_validation_session_ids))
assert not (set(fit_session_ids) & set(heldout_session_ids))
assert tuple(config['future_heldout_session_ids']) == heldout_session_ids

print('Nested moments prepared without future-session leakage.')


In [ ]:
# Fit candidate dimensions on sessions 1-16 and select using sessions 17-20 only.

cv_rows = []
cv_models = {}
selections = {}

for condition, moments in condition_moments.items():
    train_moment, train_n = moments['cv_train']
    cv_models[condition] = {}
    for q in FA_CANDIDATE_DIMS:
        print(f'CV fit: {condition}, q={q}')
        model = fit_factor_model(train_moment, q)
        cv_models[condition][q] = model
        evaluations = {
            'cv_train': (train_moment, train_n),
            **{f'validation:{s}': value for s, value in moments['cv_validation'].items()},
            'validation:pooled': moments['cv_validation_pooled'],
        }
        for evaluation, (moment, n_bins) in evaluations.items():
            cv_rows.append({
                'condition': condition,
                'q': q,
                'evaluation': evaluation,
                'n_bins': n_bins,
                'average_log_likelihood': average_log_likelihood(moment, model),
                'posterior_reconstruction_mse': posterior_reconstruction_mse(moment, model),
                'model_shared_variance_fraction': shared_variance_fraction(model),
                'n_iter': model['n_iter'],
                'converged': model['converged'],
            })

    if not all(model['converged'] for model in cv_models[condition].values()):
        raise RuntimeError(f'Non-converged candidate cannot enter selection: {condition}')
    pooled = [row for row in cv_rows
              if row['condition'] == condition and row['evaluation'] == 'validation:pooled']
    pooled = sorted(pooled, key=lambda row: row['q'])
    baseline = next(row['average_log_likelihood'] for row in pooled if row['q'] == 0)
    best_row = max(pooled, key=lambda row: row['average_log_likelihood'])
    best_gain = best_row['average_log_likelihood'] - baseline
    target_gain = PARSIMONY_GAIN_FRACTION * max(best_gain, 0.0)
    q95 = next(
        row['q'] for row in pooled
        if row['average_log_likelihood'] - baseline >= target_gain - 1e-12
    )
    selections[condition] = {
        'best_cv_q': int(best_row['q']),
        'q95_validation_gain': int(q95),
        'diagonal_validation_log_likelihood': float(baseline),
        'best_validation_log_likelihood': float(best_row['average_log_likelihood']),
        'best_validation_gain_per_bin': float(best_gain),
        'best_at_grid_maximum': bool(best_row['q'] == max(FA_CANDIDATE_DIMS)),
    }

cv_df = pd.DataFrame(cv_rows)
cv_df['delta_log_likelihood_vs_q0'] = cv_df.groupby(
    ['condition', 'evaluation']
)['average_log_likelihood'].transform(lambda values: values - values.iloc[0])
cv_df['delta_log_likelihood_per_channel'] = (
    cv_df['delta_log_likelihood_vs_q0'] / N_CHANNELS
)
cv_df.to_csv(WORK_DIR / 'cv_factor_metrics.csv', index=False)

display(pd.DataFrame(selections).T)
display(cv_df[cv_df['evaluation'] == 'validation:pooled'])


In [ ]:
# Refit every candidate on all 20 earlier sessions and evaluate the untouched future sessions.

final_models = {}
candidate_rows = []
future_rows = []
selected_loadings_rows = []
channel_rows = []
summary = {
    'run_name': RUN_NAME,
    'smoke_mode': SMOKE_MODE,
    'source_pca_run_name': PCA_RUN_NAME,
    'cv_train_session_ids': list(cv_train_session_ids),
    'cv_validation_session_ids': list(cv_validation_session_ids),
    'final_fit_session_ids': list(fit_session_ids),
    'future_heldout_session_ids': list(heldout_session_ids),
    'conditions': {},
}

for condition, moments in condition_moments.items():
    train_moment, train_n = moments['final_train']
    final_models[condition] = {}
    for q in FA_CANDIDATE_DIMS:
        print(f'Final fit: {condition}, q={q}')
        model = fit_factor_model(train_moment, q)
        final_models[condition][q] = model
        candidate_rows.append({
            'condition': condition,
            'q': q,
            'n_fit_bins': train_n,
            'train_average_log_likelihood': average_log_likelihood(train_moment, model),
            'model_shared_variance_fraction': shared_variance_fraction(model),
            'train_posterior_reconstruction_mse': posterior_reconstruction_mse(
                train_moment, model,
            ),
            'n_iter': model['n_iter'],
            'converged': model['converged'],
        })
        evaluations = {
            **moments['future'],
            'pooled': moments['future_pooled'],
        }
        for evaluation, (moment, n_bins) in evaluations.items():
            future_rows.append({
                'condition': condition,
                'q': q,
                'evaluation': evaluation,
                'n_bins': n_bins,
                'average_log_likelihood': average_log_likelihood(moment, model),
                'posterior_reconstruction_mse': posterior_reconstruction_mse(moment, model),
            })

    selected_q = selections[condition]['best_cv_q']
    selected_model = final_models[condition][selected_q]
    if not selected_model['converged']:
        raise RuntimeError(f'Selected final model did not converge: {condition}, q={selected_q}')
    loadings = ordered_loadings(selected_model)
    for factor in range(loadings.shape[1]):
        for channel in range(N_CHANNELS):
            selected_loadings_rows.append({
                'condition': condition,
                'selected_q': selected_q,
                'ordered_factor': factor + 1,
                'channel': channel,
                'loading': loadings[channel, factor],
            })
    shared_channel_variance = np.diag(loadings @ loadings.T)
    private_channel_variance = selected_model['noise_variance']
    for channel in range(N_CHANNELS):
        model_total = shared_channel_variance[channel] + private_channel_variance[channel]
        channel_rows.append({
            'condition': condition,
            'selected_q': selected_q,
            'channel': channel,
            'shared_variance': shared_channel_variance[channel],
            'private_variance': private_channel_variance[channel],
            'model_total_variance': model_total,
            'shared_fraction': shared_channel_variance[channel] / model_total,
            'empirical_train_variance': train_moment[channel, channel],
        })

candidate_df = pd.DataFrame(candidate_rows)
future_df = pd.DataFrame(future_rows)
future_df['delta_log_likelihood_vs_q0'] = future_df.groupby(
    ['condition', 'evaluation']
)['average_log_likelihood'].transform(lambda values: values - values.iloc[0])
future_df['delta_log_likelihood_per_channel'] = (
    future_df['delta_log_likelihood_vs_q0'] / N_CHANNELS
)
loadings_df = pd.DataFrame(
    selected_loadings_rows,
    columns=('condition', 'selected_q', 'ordered_factor', 'channel', 'loading'),
)
channels_df = pd.DataFrame(channel_rows)

for condition in condition_moments:
    selected_q = selections[condition]['best_cv_q']
    selected_model = final_models[condition][selected_q]
    q95 = selections[condition]['q95_validation_gain']
    q95_model = final_models[condition][q95]
    pooled = future_df[
        (future_df['condition'] == condition)
        & (future_df['evaluation'] == 'pooled')
        & (future_df['q'] == selected_q)
    ].iloc[0]
    q95_pooled = future_df[
        (future_df['condition'] == condition)
        & (future_df['evaluation'] == 'pooled')
        & (future_df['q'] == q95)
    ].iloc[0]
    summary['conditions'][condition] = {
        **selections[condition],
        'selected_model_converged': bool(selected_model['converged']),
        'selected_model_iterations': int(selected_model['n_iter']),
        'selected_model_shared_variance_fraction': shared_variance_fraction(selected_model),
        'pooled_future_average_log_likelihood': float(pooled['average_log_likelihood']),
        'pooled_future_gain_vs_diagonal_per_bin': float(
            pooled['delta_log_likelihood_vs_q0']
        ),
        'pooled_future_gain_vs_diagonal_per_channel': float(
            pooled['delta_log_likelihood_per_channel']
        ),
        'pooled_future_posterior_reconstruction_mse': float(
            pooled['posterior_reconstruction_mse']
        ),
        'q95_model_converged': bool(q95_model['converged']),
        'q95_model_iterations': int(q95_model['n_iter']),
        'q95_model_shared_variance_fraction': shared_variance_fraction(q95_model),
        'q95_pooled_future_average_log_likelihood': float(
            q95_pooled['average_log_likelihood']
        ),
        'q95_pooled_future_gain_vs_diagonal_per_bin': float(
            q95_pooled['delta_log_likelihood_vs_q0']
        ),
        'q95_pooled_future_gain_vs_diagonal_per_channel': float(
            q95_pooled['delta_log_likelihood_per_channel']
        ),
        'q95_pooled_future_posterior_reconstruction_mse': float(
            q95_pooled['posterior_reconstruction_mse']
        ),
    }

candidate_df.to_csv(WORK_DIR / 'final_candidate_model_metrics.csv', index=False)
future_df.to_csv(WORK_DIR / 'future_session_metrics.csv', index=False)
loadings_df.to_csv(WORK_DIR / 'selected_factor_loadings.csv', index=False)
channels_df.to_csv(WORK_DIR / 'selected_channel_variance_partition.csv', index=False)
(WORK_DIR / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n')

model_payload = {}
for condition in condition_moments:
    q = selections[condition]['best_cv_q']
    model_payload[f'{condition}__selected_q'] = np.asarray(q, dtype=np.int64)
    model_payload[f'{condition}__loadings'] = final_models[condition][q]['loadings']
    model_payload[f'{condition}__noise_variance'] = final_models[condition][q]['noise_variance']
    q95 = selections[condition]['q95_validation_gain']
    model_payload[f'{condition}__q95'] = np.asarray(q95, dtype=np.int64)
    model_payload[f'{condition}__q95_loadings'] = final_models[condition][q95]['loadings']
    model_payload[f'{condition}__q95_noise_variance'] = final_models[condition][q95]['noise_variance']
np.savez_compressed(WORK_DIR / 'selected_factor_models.npz', **model_payload)

display(pd.DataFrame(summary['conditions']).T)
display(future_df[future_df['evaluation'] == 'pooled'])


In [ ]:
# Plot validation/future likelihood, shared variance, reconstruction, and selected loadings.

labels = {
    'session_zscore': 'Session-wise z-score (transductive)',
    'train_global_zscore': 'Training-global z-score (inductive)',
}
colors = {'session_zscore': '#3366cc', 'train_global_zscore': '#dc3912'}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
for ax, condition in zip(axes, condition_moments):
    validation = cv_df[
        (cv_df['condition'] == condition) & (cv_df['evaluation'] == 'validation:pooled')
    ]
    future = future_df[
        (future_df['condition'] == condition) & (future_df['evaluation'] == 'pooled')
    ]
    ax.plot(validation['q'], validation['delta_log_likelihood_per_channel'],
            marker='o', label='CV validation')
    ax.plot(future['q'], future['delta_log_likelihood_per_channel'],
            marker='s', label='Future sessions (descriptive)')
    ax.axvline(selections[condition]['best_cv_q'], color='black', linestyle='--',
               label='CV-selected q')
    ax.axvline(selections[condition]['q95_validation_gain'], color='gray', linestyle=':',
               label='95% validation-gain q')
    ax.axhline(0, color='black', linewidth=0.7)
    ax.set_title(labels[condition])
    ax.set_xlabel('Number of factors')
    ax.set_ylabel('Log-likelihood gain vs diagonal\n(nats/bin/channel)')
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
fig.savefig(WORK_DIR / 'factor_dimension_likelihood.png', dpi=180)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
for condition in condition_moments:
    subset = candidate_df[candidate_df['condition'] == condition]
    axes[0].plot(subset['q'], subset['model_shared_variance_fraction'], marker='o',
                 color=colors[condition], label=labels[condition])
    future = future_df[
        (future_df['condition'] == condition) & (future_df['evaluation'] == 'pooled')
    ]
    axes[1].plot(future['q'], future['posterior_reconstruction_mse'], marker='o',
                 color=colors[condition], label=labels[condition])
axes[0].set_xlabel('Number of factors')
axes[0].set_ylabel('Model-implied shared variance fraction')
axes[0].set_title('Shared versus private model variance')
axes[1].set_xlabel('Number of factors')
axes[1].set_ylabel('Posterior-mean reconstruction MSE')
axes[1].set_title('Future-session latent reconstruction')
for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
fig.savefig(WORK_DIR / 'factor_variance_and_reconstruction.png', dpi=180)
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(15, 7), constrained_layout=True)
for ax, condition in zip(axes, condition_moments):
    q = selections[condition]['best_cv_q']
    loadings = ordered_loadings(final_models[condition][q])[:, :min(q, 16)].T
    if q == 0:
        ax.text(0.5, 0.5, 'Diagonal model selected (q=0)', ha='center', va='center')
        ax.set_axis_off()
        continue
    vmax = float(np.max(np.abs(loadings)))
    image = ax.imshow(loadings, aspect='auto', cmap='coolwarm', vmin=-vmax, vmax=vmax,
                      interpolation='nearest')
    ax.set_title(f'{labels[condition]}: first {loadings.shape[0]} ordered factors (selected q={q})')
    ax.set_ylabel('Factor')
    ax.set_yticks(np.arange(loadings.shape[0]), np.arange(1, loadings.shape[0] + 1))
    ax.set_xlabel('Area-6v channel index')
    fig.colorbar(image, ax=ax, shrink=0.8, label='Loading')
fig.savefig(WORK_DIR / 'selected_factor_loadings.png', dpi=180)
plt.show()


## Persist, reopen, and verify artifacts

Results are staged locally and promoted to a versioned Drive directory. Existing results are never silently replaced. If overwrite is explicitly enabled, the previous directory is moved to a timestamped backup.


In [ ]:
# Promote staged artifacts to Drive and verify every required artifact can be reopened.

import shutil
import uuid

required_artifacts = (
    'config.json',
    'provenance.json',
    'source_session_inventory.csv',
    'cv_factor_metrics.csv',
    'final_candidate_model_metrics.csv',
    'future_session_metrics.csv',
    'selected_factor_loadings.csv',
    'selected_channel_variance_partition.csv',
    'selected_factor_models.npz',
    'summary.json',
    'factor_dimension_likelihood.png',
    'factor_variance_and_reconstruction.png',
    'selected_factor_loadings.png',
)
missing_staged = [name for name in required_artifacts if not (WORK_DIR / name).is_file()]
if missing_staged:
    raise FileNotFoundError(f'Missing staged artifacts: {missing_staged}')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if OUTPUT_DIR.exists():
    if not OVERWRITE_OUTPUT:
        raise FileExistsError(f'Refusing to overwrite existing output: {OUTPUT_DIR}')
    backup_suffix = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    backup_dir = OUTPUT_DIR.with_name(f'{OUTPUT_DIR.name}_backup_{backup_suffix}')
    OUTPUT_DIR.rename(backup_dir)
    print('Moved previous output to recoverable backup:', backup_dir)

staging_dir = OUTPUT_ROOT / f'.{RUN_NAME}.staging.{uuid.uuid4().hex}'
shutil.copytree(WORK_DIR, staging_dir)
staging_dir.rename(OUTPUT_DIR)

missing_drive = [name for name in required_artifacts if not (OUTPUT_DIR / name).is_file()]
if missing_drive:
    raise FileNotFoundError(f'Drive promotion incomplete: {missing_drive}')

reopened_config = json.loads((OUTPUT_DIR / 'config.json').read_text())
reopened_summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
reopened_cv = pd.read_csv(OUTPUT_DIR / 'cv_factor_metrics.csv')
reopened_candidates = pd.read_csv(OUTPUT_DIR / 'final_candidate_model_metrics.csv')
reopened_future = pd.read_csv(OUTPUT_DIR / 'future_session_metrics.csv')
reopened_loadings = pd.read_csv(OUTPUT_DIR / 'selected_factor_loadings.csv')
reopened_channels = pd.read_csv(OUTPUT_DIR / 'selected_channel_variance_partition.csv')
with np.load(OUTPUT_DIR / 'selected_factor_models.npz') as reopened_models:
    reopened_model_payload = {key: reopened_models[key].copy() for key in reopened_models.files}
reopened_model_keys = set(reopened_model_payload)

if reopened_config['run_name'] != RUN_NAME or reopened_summary['run_name'] != RUN_NAME:
    raise AssertionError('Reopened run identity changed')
if tuple(reopened_config['future_heldout_session_ids']) != heldout_session_ids:
    raise AssertionError('Reopened future-session split changed')
if len(reopened_cv) != 2 * len(FA_CANDIDATE_DIMS) * 6:
    raise AssertionError(f'Unexpected CV metric row count: {len(reopened_cv)}')
if len(reopened_future) != 2 * len(FA_CANDIDATE_DIMS) * 5:
    raise AssertionError(f'Unexpected future metric row count: {len(reopened_future)}')
if len(reopened_candidates) != 2 * len(FA_CANDIDATE_DIMS):
    raise AssertionError(f'Unexpected candidate metric row count: {len(reopened_candidates)}')
if len(reopened_channels) != 2 * N_CHANNELS:
    raise AssertionError(f'Unexpected channel-partition row count: {len(reopened_channels)}')
for condition in condition_moments:
    expected_keys = {
        f'{condition}__selected_q', f'{condition}__loadings',
        f'{condition}__noise_variance', f'{condition}__q95',
        f'{condition}__q95_loadings', f'{condition}__q95_noise_variance',
    }
    if not expected_keys <= reopened_model_keys:
        raise AssertionError(f'Reopened selected model is incomplete for {condition}')
    model_specs = (
        (
            'selected',
            int(np.asarray(reopened_model_payload[f'{condition}__selected_q']).item()),
            '',
            int(reopened_summary['conditions'][condition]['best_cv_q']),
        ),
        (
            'q95',
            int(np.asarray(reopened_model_payload[f'{condition}__q95']).item()),
            'q95_',
            int(reopened_summary['conditions'][condition]['q95_validation_gain']),
        ),
    )
    for label, q, prefix, expected_q in model_specs:
        if q != expected_q or q not in FA_CANDIDATE_DIMS:
            raise AssertionError(f'Reopened {condition} {label} factor count changed')
        loadings = reopened_model_payload[f'{condition}__{prefix}loadings']
        noise = reopened_model_payload[f'{condition}__{prefix}noise_variance']
        if loadings.shape != (N_CHANNELS, q) or noise.shape != (N_CHANNELS,):
            raise AssertionError(f'Reopened {condition} {label} model has invalid shapes')
        if not np.isfinite(loadings).all() or not np.isfinite(noise).all():
            raise AssertionError(f'Reopened {condition} {label} model is nonfinite')
        if np.any(noise <= 0):
            raise AssertionError(f'Reopened {condition} {label} private variance is invalid')
expected_loading_rows = N_CHANNELS * sum(
    int(reopened_summary['conditions'][condition]['best_cv_q'])
    for condition in condition_moments
)
if len(reopened_loadings) != expected_loading_rows:
    raise AssertionError(f'Unexpected selected-loading row count: {len(reopened_loadings)}')
if not np.isfinite(reopened_cv.select_dtypes(include=[np.number]).to_numpy()).all():
    raise AssertionError('Reopened CV metrics contain nonfinite values')
if not reopened_cv['converged'].eq(True).all():
    raise AssertionError('Reopened CV metrics contain a non-converged candidate')
if not reopened_candidates['converged'].eq(True).all():
    raise AssertionError('Reopened final metrics contain a non-converged candidate')
if not np.isfinite(reopened_candidates.select_dtypes(include=[np.number]).to_numpy()).all():
    raise AssertionError('Reopened candidate metrics contain nonfinite values')
if not np.isfinite(reopened_future.select_dtypes(include=[np.number]).to_numpy()).all():
    raise AssertionError('Reopened future metrics contain nonfinite values')
if not np.isfinite(reopened_loadings.select_dtypes(include=[np.number]).to_numpy()).all():
    raise AssertionError('Reopened selected loadings contain nonfinite values')
if not np.isfinite(reopened_channels.select_dtypes(include=[np.number]).to_numpy()).all():
    raise AssertionError('Reopened channel partitions contain nonfinite values')

print('Artifact reopen checks passed.')
print('Saved:', OUTPUT_DIR)
print(json.dumps(reopened_summary['conditions'], indent=2))


## Reading the result

Start with the session-wise condition.

1. `best_cv_q` asks how many factors maximize likelihood in four earlier validation sessions.
2. `q95_validation_gain` asks how many factors recover most of the benefit over independent channels; its model and future metrics are saved alongside the maximum-likelihood selection.
3. `selected_model_shared_variance_fraction` is the model's shared/private variance partition; it is **not PCA explained variance**.
4. The future likelihood gain asks whether the shared covariance learned from all 20 earlier sessions transfers to the four untouched future days.

A small useful `q95_validation_gain` with substantial shared variance would support a compact shared signal hidden by private channel noise. A large selected dimension, little shared variance, or poor future transfer would argue against a single stationary linear FA model. Do not use the future-session curve to revise the selected dimension after seeing the result.

FA loadings are rotation-indeterminate: equally valid rotations can produce different-looking factor heatmaps. Interpret the shared covariance and per-channel shared/private partitions, not the identity of an individual plotted factor.


In [ ]:
# Final Colab teardown. Run only after artifact reopen checks pass.

if IN_COLAB:
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    runtime.unassign()
else:
    print('Local run: no Colab Drive mount or runtime to release.')
